<a href="https://colab.research.google.com/github/ArushRastogi47/10x.ai/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArushRastogi47/10x.ai/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.
***Unit of analysis:**

Unit of analysis: one row in my frame = one content item (content_hash_id, one page) owned by one client, as of the decision date 2026-03-31 — snapshot-style, the same unit as my lane's refresh queue. The warehouse fact table underneath is finer-grained: one row per report_date × client × content (a page-day), which my frame aggregates over fixed windows.
Feature window: 2026-02-01 → 2026-03-31 (trailing 60 days; all of it is history at the decision moment). Outcome window: 2026-04-01 → 2026-04-30 (the following month, from the month=2026-04 partition). I iterate on the mid-panel month 2026-03; the _sample table (June 2026) stays sealed — it is the natural outcome window of any past→future label, so developing label logic there would be a leakage trap.



In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Adjust the path pattern to match what list_repo_files actually showed
import os
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
[f for f in files if "month=2026-03" in f][:10], len(files)

(['fact_content_daily_performance/month=2026-03/data_0.parquet'], 24)

In [3]:
import os
print(os.getcwd())  # Colab normally shows /content

try:
    from google.colab import userdata
    print("google.colab import: OK")
    print("secret prefix:", userdata.get("HF_TOKEN")[:4] + "...")  # should print "hf_..."
except Exception as e:
    print("problem:", type(e).__name__, "-", e)

/content
google.colab import: OK
secret prefix: hf_N...


In [1]:
import os
print("HF_TOKEN" in os.environ)

False


In [2]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [3]:
from huggingface_hub import whoami
print(whoami()["name"])  # your HF username

ArushRastogi


In [4]:
from huggingface_hub import HfApi
api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
print(len(files), "files")
[f for f in files if "month=2026-03" in f][:10]

24 files


['fact_content_daily_performance/month=2026-03/data_0.parquet']

In [7]:
import duckdb
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, PROVIDER credential_chain)")  # or reuse the env-var secret cell
rel = "hf://datasets/FlyRank/internship-warehouse"

con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [8]:
con.sql(f"""
SELECT
  COUNT(*) AS rows,
  MIN(report_date) AS min_date,
  MAX(report_date) AS max_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [9]:
con.sql(f"""
SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet') LIMIT 3
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [10]:
# Grain probe on the iteration month: expect 0 rows back
con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
GROUP BY 1, 2, 3
HAVING c > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,c


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context (never model inputs): client_hash_id, content_hash_id — pseudonymized IDs, used for grouping/joining/splitting only; report_date and month — define windows; client_has_gsc / client_has_ga4 — client-level descriptors, used as filters only.
Features (5 max), all from the trailing window ending 2026-03-31:
impr_trailing30 — March impressions sum. Knowable at the decision moment because March has already happened by 2026-03-31.
position_mean_60d — mean gsc_avg_position over Feb–Mar. Knowable because it is averaged history, not a forecast.
ctr_60d — Σclicks/Σimpressions over Feb–Mar. Knowable because both counts are past measurements.
momentum_30d — March impressions ÷ February impressions. Knowable because both months are in the past; it measures observed direction, not the label.
days_seen_60d — distinct days the page appeared with gsc_data_available IS TRUE. Knowable because it is a count of past observations.
Label / proxy: label_declined_apr = 1 if April 2026 impressions < March impressions, else 0 (pages absent in April count as 0 impressions → declining). Computed from the outcome window — never a feature.
Excluded: fact_content_query_90d — its fixed 90-day window overlaps my outcome window (it covers April), so query-level aggregates would smuggle future information into features; per the data skill, only *_prev30 columns would be safe, so I exclude the whole table for now. I also exclude gsc_sum_position: summing positions across page-days is meaningless arithmetic (the skill's repeated-column trap), and gsc_avg_position already carries the information.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# April partition exists and covers the outcome window (mechanics check only)
[f for f in files if "month=2026-04" in f]

['fact_content_daily_performance/month=2026-04/data_0.parquet']

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three facts, each with a query on month=2026-03: (1) the grain holds, (2) the slice's row count and date span, (3) availability — how many rows survive IS TRUE filtering.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT COUNT(*) AS rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
# observed: 9,841,378 rows, 2026-03-01 -> 2026-03-31
# directional note: docs put the full table at ~78.8M rows over ~17 months and the
# June sample at ~11.7M rows, so later months carrying more rows than the average
# month is consistent with a panel that accrues content over time.

,rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [13]:
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_rows,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_rows,
  ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS pct_gsc
FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_rows,ga4_rows,pct_gsc
0,9841378,3611061,413966,36.7


In [15]:
import numpy as np

FEATURE_SQL = f"""
WITH base AS (
  SELECT client_hash_id, content_hash_id, report_date,
         gsc_impressions, gsc_clicks, gsc_avg_position
  FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-0[2-4]/*.parquet')
  WHERE gsc_data_available IS TRUE
),
feat AS (
  SELECT client_hash_id, content_hash_id,
    SUM(gsc_impressions) FILTER (report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31') AS impr_trailing30,
    SUM(gsc_impressions) FILTER (report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28') AS impr_prior30,
    SUM(gsc_clicks)      FILTER (report_date BETWEEN DATE '2026-02-01' AND DATE '2026-03-31') AS clicks_60d,
    SUM(gsc_impressions) FILTER (report_date BETWEEN DATE '2026-02-01' AND DATE '2026-03-31') AS impr_60d,
    AVG(gsc_avg_position) FILTER (report_date BETWEEN DATE '2026-02-01' AND DATE '2026-03-31') AS position_mean_60d,
    COUNT(DISTINCT report_date) AS days_seen_60d
  FROM base
  WHERE report_date BETWEEN DATE '2026-02-01' AND DATE '2026-03-31'
  GROUP BY 1, 2
),
outcome AS (
  SELECT content_hash_id,
         SUM(gsc_impressions) AS impr_apr
  FROM base
  WHERE report_date BETWEEN DATE '2026-04-01' AND DATE '2026-04-30'
  GROUP BY 1
)
SELECT f.*, COALESCE(o.impr_apr, 0) AS impr_apr,
       (COALESCE(o.impr_apr, 0) < f.impr_trailing30)::INT AS label_declined_apr
FROM feat f LEFT JOIN outcome o USING (content_hash_id)
"""

ff = con.sql(FEATURE_SQL).df()
ff["ctr_60d"] = ff["clicks_60d"] / ff["impr_60d"].replace(0, np.nan)
ff["momentum_30d"] = ff["impr_trailing30"] / ff["impr_prior30"].replace(0, np.nan)
ff.to_csv("w03_feature_frame_2026_03.csv", index=False)  # local cache only — do NOT commit
print(len(ff), "pages in frame;", ff["label_declined_apr"].mean().round(3), "observed April-decline rate")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

196059 pages in frame; 0.634 observed April-decline rate


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

FEATURES = ["impr_trailing30", "position_mean_60d", "ctr_60d", "momentum_30d", "days_seen_60d"]
m = ff.dropna(subset=FEATURES + ["label_declined_apr"]).reset_index(drop=True)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr, te = next(gss.split(m[FEATURES], m["label_declined_apr"], groups=m["client_hash_id"]))

def fit_and_score(feat_cols):
    model = LogisticRegression(max_iter=2000)
    model.fit(m.loc[tr, feat_cols], m.loc[tr, "label_declined_apr"])
    return precision_at_k(model.predict_proba(m.loc[te, feat_cols])[:, 1],
                          m.loc[te, "label_declined_apr"].values, 50)

honest_p50 = fit_and_score(FEATURES)
leaky_p50  = fit_and_score(FEATURES + ["impr_apr"])   # label-derived column added ON PURPOSE
# then remove impr_apr and keep only the honest number
print(f"baseline label rate: {m.loc[te,'label_declined_apr'].mean():.3f}")
print(f"Precision@50 honest : {honest_p50:.3f}")
print(f"Precision@50 leaky  : {leaky_p50:.3f}  <- jumps toward perfect because impr_apr IS the outcome window")

baseline label rate: 0.700
Precision@50 honest : 0.540
Precision@50 leaky  : 1.000  <- jumps toward perfect because impr_apr IS the outcome window


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Named limitation — unbalanced panel. dim_clients records each client's gsc_data_start, and clients entered the panel at different times. A fixed 60-day feature window therefore covers very different real history depths per client — for some clients it is their entire history. Consequence (directional, decision-support only): features like days_seen_60d and impr_trailing30 are not directly comparable across clients; comparing them as absolutes would bias the queue toward deep-history clients. A per-client normalization belongs in a later week.
Supporting observation: GA4 columns are mostly <NA> for GSC-only clients (client_has_ga4 = FALSE), so after IS TRUE filtering this slice is GSC-only by construction — the contract says nothing about on-page engagement.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.